# fusedkernels on a T4: correctness, then achieved bandwidth vs PyTorch

GPU session required. ~3 min install, ~2 min tests, ~5 min benchmark.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv

In [ ]:
![ -d /kaggle/working/fusedkernels-repo ] || git clone -q https://github.com/NomanJafar/fusedkernels.git /kaggle/working/fusedkernels-repo
%pip install -q "/kaggle/working/fusedkernels-repo[dev]"   # non-editable on purpose: editable .pth is inert in a running kernel
import os; os.makedirs("/kaggle/working/results", exist_ok=True); os.chdir("/kaggle/working")
import torch, triton; print("torch", torch.__version__, "triton", triton.__version__, "cuda", torch.cuda.is_available())

## 1. Correctness on the GPU (same tests as the CPU interpreter run)

In [ ]:
!cd /kaggle/working/fusedkernels-repo && python -m pytest -q tests 2>&1 | tail -5

## 2. Benchmark: eager PyTorch vs torch.compile(unfused) vs Triton, six shapes, FP16

Numbers are median kernel time with L2 flushed between runs, converted to achieved GB/s against the T4's 320 GB/s peak.

In [ ]:
!python -m fusedkernels.bench --out results/bench.json --dtype float16

In [ ]:
from fusedkernels.plots import plot_bench
from IPython.display import Image, display
for p in plot_bench("results/bench.json", "results"):
    display(Image(str(p)))

## 3. Block size and num_warps on one shape

In [ ]:
import json, torch, triton.testing
from fusedkernels.softmax import softmax, _num_warps
x = torch.randn(4096, 4096, device="cuda", dtype=torch.float16)
rows = []
for block in (4096, 8192, 16384):
    ms = triton.testing.do_bench(lambda: softmax(x, block_size=block), return_mode="median")
    rows.append({"block": block, "num_warps": _num_warps(block), "ms": ms, "gb_s": 2 * x.numel() * 2 / (ms / 1000) / 1e9})
    print(rows[-1])
json.dump(rows, open("results/softmax_block_sweep.json", "w"), indent=1)

In [ ]:
!cd results && zip -qr ../results.zip . && ls -la ../results.zip